In [0]:
deliveries = spark.read.table('data.IPLdata.deliveries')
matches = spark.read.table('data.IPLdata.matches')
players = spark.read.table('data.IPLdata.players')
seasons = spark.read.table('data.IPLdata.seasons')
import pyspark.sql.functions as f
from pyspark.sql.window import Window

In [0]:
deliveries.printSchema()
matches.printSchema()
players.printSchema()
seasons.printSchema()

In [0]:
# Count total deliveries in each match
deliveries_per_match = deliveries.groupBy('match_id').count()
display(deliveries_per_match)

In [0]:
# Find total runs scored by each batting_team
runs_by_team = deliveries.groupBy('batting_team').agg(f.sum('total_runs').alias('total_runs'))
display(runs_by_team)

In [0]:
# Count total wickets taken by each bowler
wickets_by_bowler = deliveries.filter(deliveries['is_wicket'] == True).groupBy('bowler').count()
display(wickets_by_bowler)

In [0]:
# Find number of matches played in each season
matches_per_season = matches.groupBy('season').count()
display(matches_per_season)

In [0]:
# Get total number of players by nationality
players_by_nationality = players.groupBy('nationality').count()
display(players_by_nationality)

In [0]:
# Top 5 batsmen with highest total runs
top_batsmen = deliveries.groupBy('striker').agg(f.sum('batsman_runs').alias('total_runs')) \
    .orderBy(f.desc('total_runs')).limit(5)
display(top_batsmen)

In [0]:
# Calculate strike rate of each batsman
batsman_stats = deliveries.groupBy('striker').agg(
    f.sum('batsman_runs').alias('total_runs'),
    f.count('ball').alias('balls_faced')
).withColumn('strike_rate', (f.col('total_runs') / f.col('balls_faced')) * 100)
display(batsman_stats)

In [0]:
# Bowlers with highest number of wickets
wickets_by_bowler = deliveries.filter(deliveries['is_wicket'] == True) \
    .groupBy('bowler').count().orderBy(f.desc('count'))
display(wickets_by_bowler)

In [0]:
# Matches where first innings score > 180
first_innings_scores = deliveries.filter(deliveries['innings'] == 1) \
    .groupBy('match_id').agg(f.sum('total_runs').alias('first_innings_score')) \
    .filter(f.col('first_innings_score') > 180)
matches_high_score = first_innings_scores.join(matches, 'match_id')
display(matches_high_score)

In [0]:
# Player_of_match count for each player
pom_count = matches.groupBy('player_of_match').count().orderBy(f.desc('count'))
display(pom_count)

In [0]:
#Join deliveries + matches → total runs per season
df = deliveries.join(matches, deliveries.match_id == matches.match_id)
total= df.groupBy('season').agg(f.sum('total_runs').alias('total_runs'))
display(total)

In [0]:
# Top batsman in each season
df = deliveries.join(matches, deliveries.match_id == matches.match_id)
season_batsman_runs = df.groupBy('season', 'striker').agg(f.sum('batsman_runs').alias('total_runs'))
w = Window.partitionBy('season').orderBy(f.desc('total_runs'))
top_batsman_per_season = season_batsman_runs.withColumn('rank', f.rank().over(w)).filter(f.col('rank') == 1)\
    .select('season', 'striker', 'total_runs')
display(top_batsman_per_season)

In [0]:
# Winning team with total runs scored in that match
match_runs = deliveries.groupBy('match_id').agg(f.sum('total_runs').alias('total_runs'))
winning_team_runs = matches.join(match_runs, 'match_id').select('match_id', 'winner', 'total_runs')
display(winning_team_runs)

In [0]:
# City-wise average first innings score
first_innings = deliveries.filter(deliveries['innings'] == 1).groupBy('match_id').agg(f.sum('total_runs').alias('first_score'))
city_scores = first_innings.join(matches, 'match_id').groupBy('city')\
    .agg(f.round(f.avg('first_score'),2).alias('avg_score'))
display(city_scores)

In [0]:
# Players who never got player_of_match
all_players = players.select('player_name')
pom_players = matches.select('player_of_match').distinct()
never_pom = all_players.join(pom_players, all_players.player_name == pom_players.player_of_match, 'left_anti')
display(never_pom)

In [0]:
# Batsmen with average runs per match > 30
batsman_match_runs = deliveries.groupBy('striker', 'match_id').agg(f.sum('batsman_runs').alias('runs'))
batsman_avg = batsman_match_runs.groupBy('striker').agg(
    f.avg('runs').alias('avg_runs_per_match'),
    f.count('match_id').alias('matches_played')
).filter(f.col('avg_runs_per_match') > 30)
display(batsman_avg)

In [0]:
# Economy rate of each bowler
bowler_stats = deliveries.groupBy('bowler').agg(
    f.sum('total_runs').alias('runs_conceded'),
    f.count('ball').alias('balls_bowled')
).withColumn('economy_rate', (f.col('runs_conceded') / (f.col('balls_bowled') / 6)))
display(bowler_stats.select('bowler', 'economy_rate'))

In [0]:
# Matches where chasing team won
chasing_wins = matches.filter(matches['winner'] == matches['team2'])
display(chasing_wins.select('match_id', 'season', matches['team2'].alias('chasing_team'), 'winner'))

In [0]:
# Players who scored 50+ runs in a match
batsman_match_50 = batsman_match_runs.filter(f.col('runs') >= 50)
display(batsman_match_50.select('striker', 'match_id', 'runs'))

In [0]:
# Highest partnership (striker + non_striker) in a match
partnership = deliveries.groupBy('match_id', 'striker', 'non_striker').agg(f.sum('total_runs').alias('partnership_runs'))
w = Window.partitionBy('match_id').orderBy(f.desc('partnership_runs'))
highest_partnership = partnership.withColumn('rank', f.rank().over(w)).filter(f.col('rank') == 1)
display(highest_partnership.select('match_id', 'striker', 'non_striker', 'partnership_runs'))

In [0]:
# Rank batsmen by total runs within each season
df = deliveries.join(matches, deliveries.match_id == matches.match_id)
season_batsman_runs = df.groupBy('season', 'striker').agg(f.sum('batsman_runs').alias('total_runs'))
w = Window.partitionBy('season').orderBy(f.desc('total_runs'))
season_batsman_rank = season_batsman_runs.withColumn('rank', f.rank().over(w))
display(season_batsman_rank)

In [0]:
# Find cumulative runs scored by each batsman over matches
batsman_match_runs = deliveries.groupBy('striker', 'match_id').agg(f.sum('batsman_runs').alias('runs'))
w = Window.partitionBy('striker').orderBy('match_id').rowsBetween(Window.unboundedPreceding, Window.currentRow)
batsman_cum_runs = batsman_match_runs.withColumn('cumulative_runs', f.sum('runs').over(w))
display(batsman_cum_runs)

In [0]:
# Get top scorer per match using window function
w = Window.partitionBy('match_id').orderBy(f.desc('runs'))
top_scorer_per_match = batsman_match_runs.withColumn('rank', f.rank().over(w)).filter(f.col('rank') == 1)
display(top_scorer_per_match.select('match_id', 'striker', 'runs'))

In [0]:
# Find consecutive matches won by each team
matches_sorted = matches.orderBy('team1', 'team2', 'season', 'date')
w = Window.partitionBy('winner').orderBy('season', 'date')

matches_with_prev = matches_sorted.withColumn('prev_match_id', f.lag('match_id').over(w))
matches_with_prev = matches_with_prev.withColumn('consecutive_win', f.when(f.col('prev_match_id').isNotNull(), 1).otherwise(0))
display(matches_with_prev.select('match_id', 'winner', 'consecutive_win'))

In [0]:
# Find players who improved performance season over season
season_batsman_avg = df.groupBy('season', 'striker').agg(f.avg('batsman_runs').alias('avg_runs'))
w= Window.partitionBy('striker').orderBy('season')
season_batsman_avg = season_batsman_avg.withColumn('prev_avg', f.lag('avg_runs').over(w))
improved_players = season_batsman_avg.filter(f.col('prev_avg').isNotNull() & (f.col('avg_runs') > f.col('prev_avg')))
display(improved_players.select('season', 'striker', 'avg_runs', 'prev_avg'))